In [13]:
from google.colab import drive
drive.mount('/content/drive')

dir_ = '/content/drive/MyDrive/Cohand/Minh học data/Kỳ 3/NLP/'
raw_data = dir_ + 'data/raw/'
processed_data = dir_ + 'data/processed/'
requirment_file = dir_ + 'requirements.txt'
output = dir_ + 'output/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [15]:
def save_report_to_csv(y_true, y_pred, target_names, model_name, filename=None):
    """
    Lưu classification report ra file CSV.
    """
    report_dict = classification_report(y_true, y_pred, target_names=target_names, zero_division=0, output_dict=True)

    df_report = pd.DataFrame(report_dict).transpose()

    df_report['Model'] = model_name
    cols = ['Model'] + [col for col in df_report.columns if col != 'Model']
    df_report = df_report[cols]

    if filename is None:
        clean_model_name = model_name.replace(' ', '_').replace('+', '').lower()
        filename = f"report_{clean_model_name}.csv"

    df_report.to_csv(output + filename, index=True, index_label='Label')
    print(f"Saved the report of [{model_name}] into: {filename}")

    return df_report

In [16]:
df = pd.read_csv(processed_data+'processed.csv')
df = df[df['processed_data'].notna()]
print(df.shape)
df.head(5)

(15778, 11)


,data,stayingpower,texture,smell,price,others,colour,shipping,packing,processed_data,labels
0,Công dụng: tốt\r\nKết cấu: đẹp\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng tốt kết_cấu đẹp độ bền màu lâu,"['stayingpower_positive', 'texture_positive']"
1,Công dụng: son môi\r\nKết cấu: khô\r\nĐộ bền m...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng son môi kết_cấu khô độ bền màu tuyệt...,"['stayingpower_positive', 'texture_positive']"
2,"Son mịn, mùi thơm nhẹ, lâu trôi.\r\nVideo+ hìn...",positive,positive,positive,NaN,NaN,NaN,NaN,NaN,son mịn mùi thơm nhẹ lâu trôi video hình_ảnh m...,"['stayingpower_positive', 'texture_positive', ..."
3,Công dụng: đánh son\r\nKết cấu: Đóng gói cẩn t...,positive,NaN,NaN,positive,NaN,NaN,negative,NaN,công_dụng đánh son kết_cấu đóng_gói cẩn_thận đ...,"['stayingpower_positive', 'price_positive', 's..."
4,Công dụng: tốt\r\nKết cấu: tốt\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,neutral,NaN,NaN,công_dụng tốt kết_cấu tốt độ bền màu tốt đóng_...,"['stayingpower_positive', 'texture_positive', ..."


In [17]:
aspects = ['stayingpower', 'texture', 'smell', 'price', 'others', 'colour', 'shipping', 'packing']

def extract_labels(row):
    labels = []
    for aspect in aspects:
        if pd.notna(row[aspect]):
            labels.append(f"{aspect}_{row[aspect].strip().lower()}")
    return labels

df['labels'] = df.apply(extract_labels, axis=1)

# Chuyển danh sách nhãn thành ma trận nhị phân
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['labels'])

X = df['processed_data'].values

print(f"Num of rows after cleaning: {len(X)}")
print("List of the label classes:", mlb.classes_)

# Split Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set shape: {X_train.shape[0]}, Test set shape: {X_test.shape[0]}")

Num of rows after cleaning: 15778
List of the label classes: ['colour_negative' 'colour_neutral' 'colour_positive' 'others_neutral'
 'packing_negative' 'packing_neutral' 'packing_positive' 'price_negative'
 'price_neutral' 'price_positive' 'shipping_negative' 'shipping_neutral'
 'shipping_positive' 'smell_negative' 'smell_neutral' 'smell_positive'
 'stayingpower_negative' 'stayingpower_neutral' 'stayingpower_positive'
 'texture_negative' 'texture_neutral' 'texture_positive']
Train set shape: 12622, Test set shape: 3156


# Logistic Regression One-vs-Rest kết hợp TF-IDF

In [18]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

log_reg = LogisticRegression(solver='liblinear', class_weight='balanced')
ovr_model = OneVsRestClassifier(log_reg)

ovr_model.fit(X_train_tfidf, y_train)

# Predict
y_pred_ovr = ovr_model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred_ovr, target_names=mlb.classes_, zero_division=0))

df_LR = save_report_to_csv(y_test, y_pred_ovr, mlb.classes_, "logistic_regression_tf_idf")

                       precision    recall  f1-score   support

      colour_negative       0.34      0.77      0.47       134
       colour_neutral       0.22      0.69      0.33       105
      colour_positive       0.89      0.91      0.90      1350
       others_neutral       0.74      0.97      0.84       480
     packing_negative       0.36      0.50      0.42        20
      packing_neutral       0.00      0.00      0.00         2
     packing_positive       0.92      0.95      0.94       600
       price_negative       0.00      0.00      0.00         7
        price_neutral       0.43      0.38      0.40         8
       price_positive       0.96      0.94      0.95       663
    shipping_negative       0.82      0.91      0.86       326
     shipping_neutral       0.23      0.57      0.33        74
    shipping_positive       0.89      0.94      0.91       666
       smell_negative       0.61      0.86      0.71        97
        smell_neutral       0.14      0.53      0.22  

# BiLSTM kết hợp Cơ chế Attention

In [20]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Attention
from tensorflow.keras.models import Model

# Tokenizer và Padding
max_words = 10000
max_len = 100 # Độ dài tối đa của câu

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len)

# 2. Building BiLSTM + Attention
inputs = Input(shape=(max_len,))
embedding = Embedding(input_dim=max_words, output_dim=128, input_length=max_len)(inputs)

# BiLSTM
lstm_out, forward_h, forward_c, backward_h, backward_c = Bidirectional(
    LSTM(64, return_sequences=True, return_state=True)
)(embedding)

# Attention Mechanism
attention_out = Attention()([lstm_out, lstm_out])

pooled_output = tf.keras.layers.GlobalAveragePooling1D()(attention_out)

# Fully connected layers
dense = Dense(64, activation='relu')(pooled_output)
dropout = Dropout(0.5)(dense)
outputs = Dense(y.shape[1], activation='sigmoid')(dropout) # Sigmoid cho multi-label

# Train
model_bilstm = Model(inputs=inputs, outputs=outputs)
model_bilstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_bilstm.fit(X_train_seq, y_train, validation_data=(X_test_seq, y_test), epochs=10, batch_size=64)

# Predict
print("Predicting...")
y_pred_probs_bilstm = model_bilstm.predict(X_test_seq)

# Áp dụng ngưỡng (threshold) để chuyển thành nhãn nhị phân
# Ngưỡng mặc định là 0.5, nếu dự đoán > 0.5 thì coi như là có nhãn đó (1), ngược lại là (0)
threshold = 0.5
y_pred_bilstm = (y_pred_probs_bilstm >= threshold).astype(int)

# Print report
print(classification_report(y_test, y_pred_bilstm, target_names=mlb.classes_, zero_division=0))

df_LR = save_report_to_csv(y_test, y_pred_bilstm, mlb.classes_, "bilstm")

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


198/198 ━━━━━━━━━━━━━━━━━━━━ 43s 203ms/step - accuracy: 0.2093 - loss: 0.3217 - val_accuracy: 0.4278 - val_loss: 0.2478
Epoch 2/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 41s 203ms/step - accuracy: 0.3649 - loss: 0.2447 - val_accuracy: 0.5409 - val_loss: 0.1975
Epoch 3/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 43s 215ms/step - accuracy: 0.5031 - loss: 0.1944 - val_accuracy: 0.5596 - val_loss: 0.1684
Epoch 4/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 40s 204ms/step - accuracy: 0.5434 - loss: 0.1706 - val_accuracy: 0.5938 - val_loss: 0.1515
Epoch 5/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 40s 201ms/step - accuracy: 0.5531 - loss: 0.1530 - val_accuracy: 0.6090 - val_loss: 0.1393
Epoch 6/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 40s 204ms/step - accuracy: 0.5635 - loss: 0.1398 - val_accuracy: 0.5545 - val_loss: 0.1272
Epoch 7/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 44s 223ms/step - accuracy: 0.5742 - loss: 0.1283 - val_accuracy: 0.5700 - val_loss: 0.1189
Epoch 8/10
198/198 ━━━━━━━━━━━━━━━━━━━━ 40s 202ms/step - accuracy: 0.5850 - loss: 0.1203 - val